# Banking Risk & Fraud Analytics
**DataSenseHQ** — Interactive EDA Notebook

Covers: Fraud Patterns · High-Risk Customers · Transaction Monitoring · Anomaly Detection · Window Functions

In [ ]:
import sys, os
sys.path.insert(0, '../python')
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from IPython.display import display

sns.set_theme(style='darkgrid')
pd.set_option('display.float_format', '{:,.2f}'.format)

DB_PATH = '../data/banking.db'
if not os.path.exists(DB_PATH):
    import subprocess
    subprocess.run(['python3', '../python/data_generator.py'], check=True)

conn = sqlite3.connect(DB_PATH)
print('Connected.')

## 1. Executive KPIs

In [ ]:
kpi = pd.read_sql_query("""
    SELECT
        (SELECT COUNT(*) FROM customers) AS customers,
        (SELECT COUNT(*) FROM branches)  AS branches,
        COUNT(*) AS total_txns,
        SUM(is_flagged) AS fraud_txns,
        ROUND(SUM(is_flagged)*100.0/COUNT(*),2) AS fraud_rate_pct,
        ROUND(SUM(CASE WHEN is_flagged=1 THEN amount END),2) AS fraud_exposure,
        ROUND(SUM(amount),2) AS total_volume,
        ROUND(AVG(amount),2) AS avg_txn_amount
    FROM transactions
""", conn).iloc[0]

print('=' * 55)
print('BANKING RISK & FRAUD — EXECUTIVE SUMMARY')
print('=' * 55)
print(f"  Customers          : {int(kpi['customers']):,}")
print(f"  Branches           : {int(kpi['branches'])}")
print(f"  Total Transactions : {int(kpi['total_txns']):,}")
print(f"  Fraud Flagged      : {int(kpi['fraud_txns']):,} ({kpi['fraud_rate_pct']:.2f}%)")
print(f"  Fraud Exposure     : ₹{kpi['fraud_exposure']:,.0f}")
print(f"  Total Volume       : ₹{kpi['total_volume']:,.0f}")
print(f"  Avg Transaction    : ₹{kpi['avg_txn_amount']:,.0f}")

## 2. Fraud Rate by Channel

In [ ]:
ch = pd.read_sql_query("""
    SELECT channel, COUNT(*) AS total,
           SUM(is_flagged) AS frauds,
           ROUND(SUM(is_flagged)*100.0/COUNT(*),2) AS fraud_rate,
           ROUND(AVG(CASE WHEN is_flagged=1 THEN amount END),2) AS avg_fraud_amt
    FROM transactions GROUP BY channel ORDER BY fraud_rate DESC
""", conn)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(ch['channel'], ch['fraud_rate'], color=sns.color_palette('Reds_r', len(ch)))
axes[0].set_title('Fraud Rate by Channel (%)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Fraud Rate (%)')
for i, (_, r) in enumerate(ch.iterrows()):
    axes[0].text(i, r['fraud_rate']+0.05, f"{r['fraud_rate']:.1f}%", ha='center', fontsize=10)

axes[1].bar(ch['channel'], ch['avg_fraud_amt'], color=sns.color_palette('OrRd', len(ch)))
axes[1].set_title('Avg Fraud Amount by Channel', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Amount (₹)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x:,.0f}'))

plt.tight_layout()
plt.show()
display(ch)

## 3. Hourly Fraud Pattern

In [ ]:
hourly = pd.read_sql_query("""
    SELECT CAST(SUBSTR(txn_time,1,2) AS INTEGER) AS hour,
           COUNT(*) AS total, SUM(is_flagged) AS frauds,
           ROUND(SUM(is_flagged)*100.0/COUNT(*),2) AS fraud_rate
    FROM transactions GROUP BY hour ORDER BY hour
""", conn)

fig, ax = plt.subplots(figsize=(14, 5))
bar_colors = ['#e74c3c' if (h >= 22 or h <= 4) else '#3498db' for h in hourly['hour']]
ax.bar(hourly['hour'], hourly['fraud_rate'], color=bar_colors)
ax.set_title('Fraud Rate by Hour — High-Risk Hours (10pm–4am) in Red', fontsize=13, fontweight='bold')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Fraud Rate (%)')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='#e74c3c', label='High-Risk (10pm-4am)'), Patch(color='#3498db', label='Normal')])
plt.tight_layout()
plt.show()

## 4. Monthly Fraud Rate Trend

In [ ]:
monthly = pd.read_sql_query("""
    SELECT strftime('%Y-%m',txn_date) AS month,
           COUNT(*) AS total, SUM(is_flagged) AS frauds,
           ROUND(SUM(is_flagged)*100.0/COUNT(*),2) AS fraud_rate,
           ROUND(SUM(amount),2) AS volume
    FROM transactions GROUP BY month ORDER BY month
""", conn)
monthly['month'] = pd.to_datetime(monthly['month'])
monthly['rolling_3m'] = monthly['fraud_rate'].rolling(3).mean()

fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)
axes[0].plot(monthly['month'], monthly['fraud_rate'], marker='o', color='#e74c3c', linewidth=2, label='Monthly')
axes[0].plot(monthly['month'], monthly['rolling_3m'], color='#c0392b', linewidth=2, linestyle='--', label='3M Rolling Avg')
axes[0].fill_between(monthly['month'], monthly['fraud_rate'], alpha=0.15, color='#e74c3c')
axes[0].axhline(y=3, color='#e67e22', linestyle=':', label='Warning (3%)')
axes[0].axhline(y=5, color='#e74c3c', linestyle=':', label='Alert (5%)')
axes[0].set_title('Monthly Fraud Rate with Alerts', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Fraud Rate (%)')
axes[0].legend(fontsize=8)

axes[1].bar(monthly['month'], monthly['volume']/1e6, width=20, color='#6c5ce7', alpha=0.7)
axes[1].set_title('Monthly Transaction Volume (₹M)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Volume (₹M)')
axes[1].xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%b %Y'))
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 5. Risk Score Distribution

In [ ]:
risk = pd.read_sql_query("""
    SELECT r.risk_label, r.risk_score, r.velocity_flag, r.large_txn_flag,
           r.odd_hour_flag, r.multi_city_flag, c.credit_score, c.account_balance
    FROM risk_scores r JOIN customers c ON r.customer_id=c.id
""", conn)

label_order = ['Critical','High','Medium','Low']
colors = {'Critical':'#e74c3c','High':'#e67e22','Medium':'#f1c40f','Low':'#27ae60'}
counts = risk['risk_label'].value_counts().reindex(label_order, fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].pie(counts, labels=counts.index, autopct='%1.1f%%',
            colors=[colors[l] for l in counts.index], startangle=90)
axes[0].set_title('Customer Risk Distribution', fontsize=13, fontweight='bold')

data = [risk[risk['risk_label']==l]['risk_score'].values for l in label_order if (risk['risk_label']==l).any()]
labels_present = [l for l in label_order if (risk['risk_label']==l).any()]
bp = axes[1].boxplot(data, patch_artist=True, labels=labels_present)
for patch, lbl in zip(bp['boxes'], labels_present):
    patch.set_facecolor(colors[lbl]); patch.set_alpha(0.75)
axes[1].set_title('Risk Score by Label', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Risk Score')
plt.tight_layout()
plt.show()

print('Risk score summary:')
display(risk.groupby('risk_label')['risk_score'].describe().round(2))

## 6. Window Function: Running Fraud Rate per Customer

In [ ]:
# Top 5 highest fraud-rate customers and their running cumulative fraud
top5_cust = pd.read_sql_query("""
    SELECT c.customer_id, COUNT(*) AS txns,
           SUM(t.is_flagged) AS fraud_txns,
           ROUND(SUM(t.is_flagged)*100.0/COUNT(*),1) AS fraud_pct
    FROM transactions t JOIN customers c ON t.customer_id=c.id
    GROUP BY c.id HAVING txns >= 15
    ORDER BY fraud_pct DESC LIMIT 5
""", conn)

print('Top 5 customers by fraud rate:')
display(top5_cust)

# Rolling window: 7-day avg volume and fraud rate
daily = pd.read_sql_query("""
    SELECT txn_date, COUNT(*) AS txns, SUM(is_flagged) AS frauds,
           ROUND(SUM(amount),2) AS volume
    FROM transactions GROUP BY txn_date ORDER BY txn_date
""", conn)
daily['date'] = pd.to_datetime(daily['txn_date'])
daily['fraud_rate'] = daily['frauds'] / daily['txns'] * 100
daily['roll7_fraud'] = daily['fraud_rate'].rolling(7).mean()
daily['roll7_vol']   = daily['volume'].rolling(7).mean()

fig, ax = plt.subplots(figsize=(16, 4))
ax.plot(daily['date'], daily['roll7_fraud'], color='#e74c3c', linewidth=2, label='7-Day Rolling Fraud Rate')
ax.axhline(y=daily['roll7_fraud'].mean(), color='navy', linestyle='--',
           label=f"Overall Avg: {daily['roll7_fraud'].mean():.2f}%")
ax.set_title('7-Day Rolling Fraud Rate (Window Function)', fontsize=13, fontweight='bold')
ax.set_ylabel('Fraud Rate (%)')
ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%b %Y'))
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 7. ML Anomaly Detection (Isolation Forest)

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import LabelEncoder

df = pd.read_sql_query("""
    SELECT t.id, t.amount, t.is_flagged,
           CAST(SUBSTR(t.txn_time,1,2) AS INTEGER) AS hour,
           t.channel, t.txn_type,
           CAST(strftime('%w',t.txn_date) AS INTEGER) AS dow,
           c.credit_score, c.account_balance
    FROM transactions t JOIN customers c ON t.customer_id=c.id
""", conn)

le = LabelEncoder()
df['channel_enc']  = le.fit_transform(df['channel'])
df['txn_type_enc'] = le.fit_transform(df['txn_type'])

features = ['amount','hour','channel_enc','txn_type_enc','dow','credit_score','account_balance']
iso = IsolationForest(n_estimators=100, contamination=0.04, random_state=42)
df['iso_score'] = iso.fit_predict(df[features].fillna(0))
df['is_anomaly'] = (df['iso_score'] == -1).astype(int)

overlap = ((df['is_anomaly']==1) & (df['is_flagged']==1)).sum()
new_catches = df['is_anomaly'].sum() - overlap
print(f"Isolation Forest anomalies : {df['is_anomaly'].sum():,}")
print(f"Overlap with rule-flagged  : {overlap:,}")
print(f"New catches (ML-only)      : {new_catches:,}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
normal = df[df['is_anomaly']==0].sample(min(3000, (df['is_anomaly']==0).sum()))
anom   = df[df['is_anomaly']==1]
axes[0].scatter(normal['amount'], normal['hour'], c='#6c5ce7', alpha=0.2, s=5, label='Normal')
axes[0].scatter(anom['amount'],   anom['hour'],   c='#e74c3c', alpha=0.5, s=15, label='Anomaly')
axes[0].set_xlabel('Amount (₹)'); axes[0].set_ylabel('Hour')
axes[0].set_title('Amount vs Hour — Anomalies Highlighted', fontsize=12, fontweight='bold')
axes[0].legend()

ch_compare = df.groupby('channel').agg(rule_rate=('is_flagged','mean'), ml_rate=('is_anomaly','mean')).reset_index()
x = np.arange(len(ch_compare))
axes[1].bar(x - 0.2, ch_compare['rule_rate']*100, 0.4, label='Rule-Based', color='#e67e22', alpha=0.85)
axes[1].bar(x + 0.2, ch_compare['ml_rate']*100,   0.4, label='Isolation Forest', color='#e74c3c', alpha=0.85)
axes[1].set_xticks(x); axes[1].set_xticklabels(ch_compare['channel'])
axes[1].set_title('Rule-Based vs ML Detection by Channel', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Detection Rate (%)')
axes[1].legend()
plt.tight_layout()
plt.show()

## 8. Z-Score Large Transaction Alerts

In [ ]:
txn = pd.read_sql_query("""
    SELECT t.txn_id, c.customer_id, t.amount, t.channel, t.txn_date, t.is_flagged
    FROM transactions t JOIN customers c ON t.customer_id=c.id
""", conn)

mean_a, std_a = txn['amount'].mean(), txn['amount'].std()
txn['z_score'] = (txn['amount'] - mean_a) / std_a

fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(txn['amount'].clip(0, 50000), bins=80, color='#6c5ce7', alpha=0.6, edgecolor='white')
ax.axvline(x=mean_a + 2.5*std_a, color='#e74c3c', linewidth=2, linestyle='--',
           label=f'z=2.5 threshold: ₹{mean_a+2.5*std_a:,.0f}')
ax.axvline(x=mean_a + 3*std_a, color='#c0392b', linewidth=2, linestyle=':',
           label=f'z=3.0 threshold: ₹{mean_a+3*std_a:,.0f}')
ax.set_title('Transaction Amount Distribution with Z-Score Thresholds', fontsize=13, fontweight='bold')
ax.set_xlabel('Amount (₹)')
ax.set_ylabel('Frequency')
ax.legend()
plt.tight_layout()
plt.show()

top_anomalies = txn[txn['z_score'] > 3].nlargest(10, 'z_score')[['txn_id','customer_id','amount','channel','txn_date','is_flagged','z_score']]
print('Top 10 highest z-score transactions:')
display(top_anomalies)

## 9. Branch Performance & Fraud Heatmap

In [ ]:
branch_df = pd.read_sql_query("""
    SELECT b.name AS branch, b.city,
           COUNT(DISTINCT c.id) AS customers,
           COUNT(t.id) AS transactions,
           ROUND(SUM(t.amount),2) AS volume,
           SUM(t.is_flagged) AS fraud_txns,
           ROUND(SUM(t.is_flagged)*100.0/COUNT(t.id),2) AS fraud_rate
    FROM branches b
    JOIN customers c ON c.branch_id=b.id
    JOIN transactions t ON t.customer_id=c.id
    GROUP BY b.id ORDER BY fraud_rate DESC
""", conn)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].barh(branch_df['branch'][::-1], branch_df['fraud_rate'][::-1],
             color=sns.color_palette('Reds', len(branch_df)))
axes[0].set_title('Fraud Rate by Branch', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Fraud Rate (%)')

axes[1].barh(branch_df['branch'][::-1], branch_df['volume'][::-1]/1e6,
             color=sns.color_palette('Blues', len(branch_df)))
axes[1].set_title('Transaction Volume by Branch (₹M)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Volume (₹M)')

plt.tight_layout()
plt.show()
display(branch_df)

In [ ]:
conn.close()
print('Analysis complete.')